# Conversion Modeling

> **📦 Optional dependencies**
>
> The plots in this article use `plotly`, which ships with the pdstools `adm` extra. Install with your favorite package manager, e.g. `uv pip install "pdstools[adm]"`.
>
> All code cells are collapsed by default. Expand them in Jupyter if you want to see how the (synthetic) data and plots are made.

Adaptive models in Pega Customer Decision Hub (CDH) usually learn from **engagement**: a customer clicks on a web banner, opens an e-mail, or accepts an offer in the contact center. Engagement is fast and plentiful, but it is a proxy. What the business really cares about is often the **conversion**: the customer actually buys the handset, opens the savings account, or takes out the mortgage.

Conversion modeling lets ADM learn from these downstream outcomes. This article takes a data science view of the approach: what a conversion is, how conversions are attributed to actions, when conversion modeling is (and is not) worth the effort, how far back to look, and how conversion and engagement propensities can be compared in one arbitration.

- [What is a conversion?](#What-is-a-conversion?)
- [Attribution](#Attribution)
- [Do you need conversion modeling?](#Do-you-need-conversion-modeling?)
- [The look-back period](#The-look-back-period)
- [Mixing conversion and engagement](#Mixing-conversion-and-engagement)
- [Summary](#Summary)

In [ ]:
# These lines are only for rendering in the docs, and are hidden through Jupyter tags
# Do not run if you're running the notebook separately

import plotly.io as pio

pio.renderers.default = "notebook_connected"

In [ ]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import polars as pl

rng = np.random.default_rng(42)

## What is a conversion?

Conversion modeling is mostly a **business problem**, not a modeling problem. Technically, a conversion is just another outcome that ADM learns from. The hard part is deciding *what* that outcome is. That choice varies a lot by industry, product, and business goal:

| Domain | Possible conversion events |
|---|---|
| Retail banking | Current or savings account opened, credit card activated (not just approved), first deposit or first card transaction, salary deposit switched |
| Lending | Mortgage or loan application submitted, application approved, loan funded or disbursed |
| Insurance | Quote requested, policy bound, first premium paid, additional coverage added |
| Telecommunications | Handset purchased, contract upgraded or renewed, add-on (roaming, data bundle) activated, customer not churning after a retention offer |
| Retail and e-commerce | Order placed, order not returned within the return window, loyalty program sign-up |
| Digital self-service | App installed and logged in, paperless statements enabled, card added to a digital wallet |

Marketers often separate **macro conversions** (the actual sale, such as a funded mortgage) from **micro conversions**: smaller steps that signal intent, such as using the mortgage calculator on the website, starting an application, booking an appointment with an advisor, or downloading a brochure. Micro conversions occur more often and sooner, so models learn faster. But they are further removed from business value. Macro conversions are closer to value but rare and slow, so models learn slowly.

A few questions the business needs to answer before turning on conversion modeling:

- **Which event counts?** "Application started", "application approved" and "loan disbursed" are three different conversions, with different volumes and delays.
- **Which actions promote it?** Conversions are usually reported against a *product* (the conversion ID), not an action. Several actions, treatments, and channels can promote the same product. CDH uses *product associations* to map a conversion to the actions that promote it.
- **Where does the signal come from?** Conversions usually happen outside CDH (in a store, a fulfillment system, a policy administration system). Someone has to send the conversion back to Pega, in real time or in batches.
- **How long can it take?** This sets how long decision data must be kept so that a late conversion can still be matched to the decision that caused it. See [The look-back period](#The-look-back-period).
- **Would it have happened anyway?** Many conversions are organic. Product purchases happen whether or not an action was shown, so not every conversion is evidence that an action worked.

Pega's documentation describes the configuration steps in detail: [Conversions in Next-Best-Actions](https://docs.pega.com/bundle/customer-decision-hub/page/customer-decision-hub/cdh-portal/conversions-next-best-actions.html).

## Attribution

Once a conversion is reported, it has to be linked to the actions that might have caused it. This is the **attribution problem**: given a conversion event, which of the earlier touchpoints (impressions of actions that promote the converted product) get credit, and how much?

Attribution is an old and largely unsolved problem in marketing. Approaches range from simple rules to data-driven methods such as Shapley values, Markov chain removal effects, and controlled uplift experiments. Pega does **not** try to solve attribution in a data-driven way. It offers a set of configurable, rule-based schemes. The decisioning architect picks the one that best matches the business view. Every touchpoint that receives credit is then used as a positive example (weighted by its credit) for the conversion model of that action.

The available schemes are:

- **First touch**: all credit goes to the first related action the customer saw.
- **Last touch**: all credit goes to the most recent related action before the conversion.
- **Linear**: credit is shared equally across all related actions.
- **Linear decay**: credit is shared across all related actions, with more weight for more recent ones. With $n$ touchpoints, touchpoint $i$ (oldest is 1) gets weight $\frac{i}{\sum_{j=1}^{n} j} = \frac{2i}{n(n+1)}$. For three touchpoints this is $\frac{1}{6}$, $\frac{2}{6}$, and $\frac{3}{6}$.
- **Custom**: you can plug in your own scheme.

Only touchpoints inside the attribution window (the look-back period) are considered. The example below shows a customer who saw four actions for the same product over two weeks and then bought it.

In [ ]:
touchpoints = ["E-mail<br>day 0", "Web<br>day 4", "App<br>day 8", "E-mail<br>day 11"]
n = len(touchpoints)
schemes = {
    "First touch": [1.0] + [0.0] * (n - 1),
    "Last touch": [0.0] * (n - 1) + [1.0],
    "Linear": [1 / n] * n,
    "Linear decay": [2 * (i + 1) / (n * (n + 1)) for i in range(n)],
}
attribution = pl.DataFrame(
    {
        "Scheme": [s for s in schemes for _ in touchpoints],
        "Touchpoint": touchpoints * len(schemes),
        "Credit": [c for credits in schemes.values() for c in credits],
    }
)

fig = px.bar(
    attribution,
    x="Touchpoint",
    y="Credit",
    facet_col="Scheme",
    color="Scheme",
    text_auto=".0%",
    title="Credit given to each touchpoint before a conversion on day 14",
    template="plotly_white",
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_yaxes(tickformat=".0%", range=[0, 1.15])
fig.update_traces(textangle=0, textposition="outside")
fig.update_xaxes(title_text="", tickangle=0, tickfont_size=10)
fig.update_layout(showlegend=False, height=420, margin=dict(t=90))
fig.show()

The choice of scheme matters less than you might think when customers mostly see a single action for a product. It matters more when products are promoted through many actions and channels. In that case first and last touch reward very different parts of the journey (awareness vs. closing), while linear and linear decay spread the credit.

## Do you need conversion modeling?

Conversion modeling comes with real cost: an extra integration to report conversions, decision data that has to be kept for much longer, and models that learn from far fewer and much later signals. So before investing, check whether it would actually change any decisions.

The key point is that **CDH arbitration only cares about the ranking of actions**, not the absolute propensity values. If an action's conversion rate is roughly a fixed fraction of its engagement rate, then ranking actions by engagement already gives (almost) the same order as ranking them by conversion. In that case the engagement models already do the job, and the cheaper and faster engagement signal is the better choice.

You can check this with data you probably already have: for every action, compute the engagement rate (clicks or accepts per impression) and the conversion rate (conversions per impression) over the same period, and plot one against the other. The example below uses synthetic data for 40 actions.

In [ ]:
n_actions = 40
impressions = rng.integers(20_000, 250_000, n_actions)
true_engagement = rng.lognormal(mean=np.log(0.02), sigma=0.55, size=n_actions)
true_conversion = true_engagement * 0.10 * rng.lognormal(mean=0.0, sigma=0.22, size=n_actions)
groups = rng.choice(["Cards", "Loans", "Savings", "Insurance"], n_actions)

actions = pl.DataFrame(
    {
        "Action": [f"Action {i + 1:02d}" for i in range(n_actions)],
        "Group": groups,
        "Impressions": impressions,
        "Engagements": rng.binomial(impressions, true_engagement),
        "Conversions": rng.binomial(impressions, true_conversion),
    }
).with_columns(
    EngagementRate=pl.col("Engagements") / pl.col("Impressions"),
    ConversionRate=pl.col("Conversions") / pl.col("Impressions"),
)

# Least squares fit through the origin: conversion rate = slope * engagement rate
slope = (actions["EngagementRate"] * actions["ConversionRate"]).sum() / (actions["EngagementRate"] ** 2).sum()
stats = actions.select(
    Pearson=pl.corr("EngagementRate", "ConversionRate"),
    Spearman=pl.corr("EngagementRate", "ConversionRate", method="spearman"),
)
pearson, spearman = stats.row(0)

fig = px.scatter(
    actions,
    x="EngagementRate",
    y="ConversionRate",
    size="Impressions",
    color="Group",
    hover_name="Action",
    hover_data={"Impressions": ":,", "EngagementRate": ":.2%", "ConversionRate": ":.3%"},
    template="plotly_white",
    title=(
        "Engagement vs. conversion rate per action<br>"
        f"<sup>Pearson r = {pearson:.2f}, Spearman rank correlation = {spearman:.2f}</sup>"
    ),
)
x_max = actions["EngagementRate"].max() * 1.05
fig.add_trace(
    go.Scatter(
        x=[0, x_max],
        y=[0, slope * x_max],
        mode="lines",
        line=dict(dash="dash", color="grey"),
        name=f"Conversion ≈ {slope:.1%} of engagement",
    )
)
fig.update_xaxes(tickformat=".1%", title_text="Engagement rate (engagements / impressions)", rangemode="tozero")
fig.update_yaxes(tickformat=".2%", title_text="Conversion rate (conversions / impressions)", rangemode="tozero")
fig.update_layout(height=550, margin=dict(l=90), legend_title_text="")
fig.show()

Each dot is an action, sized by its number of impressions. Here the dots lie close to a straight line through the origin: conversions are about one tenth of engagements, for every action. The rank correlation is high. Ranking actions by engagement propensity therefore gives almost the same order as ranking by conversion propensity, and conversion modeling would add little.

How to read such a plot for your own data:

- **Points close to a line through the origin**: engagement is a good proxy for conversion. Stick with engagement models. They learn faster and need no extra integration.
- **A wide spread, or clear clusters with different slopes**: some actions get a lot of clicks that do not turn into sales (think clickbait, or curiosity about a new product), while others convert well from few clicks. This is where conversion modeling can change the ranking and add business value.
- **Actions with engagement but no conversions** (on the x-axis): check first whether the conversions are reported and mapped to these actions before drawing conclusions.

This is an action-level view. A strong correlation across actions does not guarantee that the same holds within an action across customers. A natural next step is to bin the engagement propensities that ADM produced and check that the conversion rate also increases across the bins.

## The look-back period

Engagement usually arrives within seconds (a click) or days (an e-mail open). Conversions take longer: customers think about it, compare offers, visit a store, or wait for the next paycheck. To attribute a conversion to an action, CDH must keep the decision data (the customer's predictor values at decision time) until the conversion arrives. The **look-back period**, also called the attribution window, is how long CDH waits. It is a trade-off:

- **Too short**: genuine conversions arrive after the decision data is gone. The model never learns about them and the conversion rate is underestimated, especially for slower products.
- **Too long**: storage grows, models learn later, and more *organic* conversions (purchases that would have happened anyway, unrelated to the action) are credited to whatever action happened to be shown.

Choose the period from data: for every conversion, look up the most recent impression of a related action and compute the delay between the two. The example below uses synthetic data for eight products. Most conversions are driven by an impression and follow it within days. On top of that there is a small, constant background of organic conversions.

In [ ]:
max_days = 45
products = [f"Product {p}" for p in "ABCDEFGH"]
delays = []
for product in products:
    n_driven = rng.integers(1_500, 4_000)
    n_organic = int(n_driven * rng.uniform(0.15, 0.25))
    # Conversions driven by the impression: gamma distributed delay, mean of about 5 days
    driven = rng.gamma(shape=2.0, scale=rng.uniform(2.0, 3.0), size=n_driven)
    # Organic conversions: uniformly spread, unrelated to the impression
    organic = rng.uniform(0, max_days, size=n_organic)
    delays.append(pl.DataFrame({"Product": product, "Days": np.floor(np.concatenate([driven, organic]))}))

conversion_delays = (
    pl.concat(delays)
    .filter(pl.col("Days") < max_days)
    .group_by("Product", "Days")
    .agg(Conversions=pl.len())
    .sort("Product", "Days")
    .with_columns(Share=pl.col("Conversions") / pl.col("Conversions").sum().over("Product"))
    .with_columns(Cumulative=pl.col("Share").cum_sum().over("Product"))
)
summary = (
    conversion_delays.group_by("Days")
    .agg(
        pl.col("Share").mean().alias("Share"),
        pl.col("Share").std().alias("ShareStd"),
        pl.col("Cumulative").mean().alias("Cumulative"),
        pl.col("Cumulative").std().alias("CumulativeStd"),
    )
    .sort("Days")
)
within_10 = summary.filter(pl.col("Days") == 9)["Cumulative"].item()

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=summary["Days"],
        y=summary["Share"],
        error_y=dict(type="data", array=summary["ShareStd"], thickness=1, width=3),
        mode="lines+markers",
        name="Conversions on that day",
        marker_size=5,
    )
)
fig.add_trace(
    go.Scatter(
        x=summary["Days"],
        y=summary["Cumulative"],
        error_y=dict(type="data", array=summary["CumulativeStd"], thickness=1, width=3),
        mode="lines+markers",
        name="Cumulative",
        yaxis="y2",
        marker_size=5,
    )
)
fig.add_vline(
    x=9.5,
    line_dash="dash",
    line_color="grey",
    annotation_text=f"10 days: {within_10:.0%} of conversions",
    annotation_position="top right",
)
fig.update_layout(
    template="plotly_white",
    title="Days between last impression and conversion<br><sup>Mean and standard deviation across 8 products</sup>",
    xaxis=dict(title="Days since last impression of a related action"),
    yaxis=dict(title="Share of conversions per day", tickformat=".0%", rangemode="tozero"),
    yaxis2=dict(title="Cumulative share", tickformat=".0%", overlaying="y", side="right", range=[0, 1.05]),
    legend=dict(orientation="h", y=-0.2),
    height=550,
    margin=dict(l=90, r=120),
)
fig.show()

Most conversions happen within about 10 days of the last impression. After that, the daily share drops to a low, flat level. That flat tail is what you would expect from organic conversions that have nothing to do with the action. Extending the look-back period well beyond the point where the curve flattens mostly adds noise (and storage cost), not signal. In this example a look-back period of about 10 to 14 days is a sensible choice.

Some practical notes for running this analysis on your own data:

- Compare the delay curve against a **baseline**: conversions of customers who did *not* see a related action. The flat tail should match that baseline. Delays where the curve is clearly above the baseline are where the actions have an effect.
- Look at the curves **per product or product group**. A credit card and a mortgage can have very different delays, and the look-back period has to cover the slowest product you want to model.
- Dates without a time component make same-day ordering ambiguous. If the impression and the conversion are on the same day, you may not know which came first.

## Mixing conversion and engagement

In practice, not every action has a conversion model that can be used. Some actions have no conversion at all (a "happy birthday" message, a service notification). Others are new, and their conversion model has not seen enough positives yet. These actions keep using their engagement model. The pre-arbitration set of actions therefore contains a mix: some actions have a conversion propensity, others an engagement propensity. Since conversion rates are typically an order of magnitude lower than engagement rates, these numbers cannot be compared directly.

### Two propensities per action

For every action $a$ and customer $x$, CDH runs two adaptive models side by side:

- the **engagement model**, which estimates $p^{E}_{a}(x) = P(\text{engage} \mid \text{impression}, x)$ and is trained on clicks or accepts, as usual;
- the **conversion model**, which estimates $p^{C}_{a}(x) = P(\text{convert} \mid \text{impression}, x)$ and is trained on the attributed conversions, where each impression gets a weight between 0 and 1 from the attribution scheme.

The strategy uses the conversion propensity when the conversion model is mature enough (and the customer is not in a control group that keeps using engagement, see below). Otherwise it uses the engagement propensity.

### Scaling

To make a conversion propensity comparable to engagement propensities, CDH multiplies it by a per-action scaling factor: the ratio between the action's overall engagement rate and its overall conversion rate:

$$
s_a = \frac{\bar{p}^{E}_{a}}{\bar{p}^{C}_{a}} = \frac{\text{engagements}_a / \text{impressions}_a}{\text{conversions}_a / \text{impressions}_a} = \frac{\text{engagements}_a}{\text{conversions}_a}
\qquad\qquad
\tilde{p}^{C}_{a}(x) = s_a \cdot p^{C}_{a}(x)
$$

Example: an action for a handset has an overall engagement rate of 1.6% and an overall conversion rate of 0.2%, so $s_a = 8$. For a particular customer the conversion model gives 0.15%, which scales to $8 \times 0.15\% = 1.2\%$. That can now be compared with the 1.1% engagement propensity of another action, for which no conversion model is available yet.

The scaling factor is the same for all customers. So it does not change the ranking of customers *within* an action. It only puts the action on the same scale as the others. If engagement and conversion are as closely related as in the scatter plot above, $s_a$ is about 10 for every action, and the scaled conversion propensity is close to the engagement propensity. That is another way of seeing that conversion modeling then changes little.

### It is really the value that is scaled

The CDH priority multiplies the propensity with the action's value (along with context weights and business levers, which we leave out here):

$$
\text{Priority}_a(x) = p_a(x) \cdot V_a
$$

Arbitration compares actions on the expected value **per impression**. For an engagement-driven action, $V_a$ is effectively the value of an engagement. For a conversion-driven action, the natural value is the value of a conversion, $V^{C}_{a}$, for example the margin on the product, and the expected value of an impression is $p^{C}_{a}(x) \cdot V^{C}_{a}$.

Reorder the scaled priority and the scaling factor ends up on the value instead of on the propensity:

$$
\underbrace{\left(s_a \cdot p^{C}_{a}(x)\right)}_{\text{scaled propensity}} \cdot V_a
\;=\;
p^{C}_{a}(x) \cdot \underbrace{\left(s_a \cdot V_a\right)}_{V^{C}_{a}}
\qquad\Longleftrightarrow\qquad
V_a = V^{C}_{a} \cdot \frac{\text{conversions}_a}{\text{engagements}_a}
$$

So the scaling does not really "inflate" the conversion propensity. It translates the value of a conversion (large: a product sale) into the value of a single engagement (much smaller: only about one in $s_a$ engagements turns into a sale). Everything in the arbitration is then expressed in the same unit, the value of an engagement, and conversion-driven and engagement-driven actions compete fairly.

### Test and learn

Optimizing for conversion does not always produce a better business result than optimizing for engagement. The engagement models learn much faster, and the conversion signal can be sparse, delayed, or noisy. The strategy template therefore allows a part of the customers (25% by default, selected by customer ID) to keep using engagement models only. Comparing conversions in both groups shows whether conversion modeling actually pays off. The look-back period and attribution scheme can be tuned based on these results.

## Summary

- **What a conversion is depends on the business.** It can be a purchase, an activation, an application, or a micro conversion such as using a mortgage calculator. Decide this first, together with how conversions are reported and which actions promote which products.
- **Attribution** links a conversion to earlier impressions. CDH offers first touch, last touch, linear, linear decay, and custom schemes. It does not try to solve attribution in a data-driven way.
- **Check whether you need it.** If conversion rates are roughly proportional to engagement rates across actions, the ranking hardly changes and engagement models are the cheaper, faster choice.
- **Pick the look-back period from data**: long enough to capture most of the impression-driven conversions, but not so long that organic conversions and storage costs dominate.
- **Scaling** puts conversion and engagement propensities on the same scale. You can also read it as converting the value of a conversion into the value of an engagement, so all actions are compared in the same unit.